In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('..')
from src import functions as fc

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier

In [3]:
df_modeling = fc.load_data_clean("bank_final.csv")

Buscando archivo en: /Users/jbp/Desktop/IRONHACK/SEMANA7/ML_project/data/cleaned/bank_final.csv


In [4]:
features = df_modeling.drop(columns=["target", "duration"])
target = df_modeling["target"]

In [5]:
x_train, x_test, y_train, y_test = train_test_split(features, target, test_size=0.20, random_state=0)

Bagging Classifier:

In [14]:
bagging_reg = BaggingClassifier(DecisionTreeClassifier(max_depth=10), n_estimators=100, max_samples = 0.8, random_state=42)

De nuevo reducimos a 10 el max depth para evitar el overfitting y en max samples le indicamos que coja el 80%. Añadimos el random_state para que no varie el resultado cada vez que le damos a run.

In [15]:
bagging_reg.fit(x_train, y_train)

,estimator,DecisionTreeC...(max_depth=10)
,n_estimators,100
,max_samples,0.8
,max_features,1.0
,bootstrap,True
,bootstrap_features,False
,oob_score,False
,warm_start,False
,n_jobs,None
,random_state,42
,verbose,0


In [16]:
pred = bagging_reg.predict(x_test)

In [17]:
accuracy = bagging_reg.score(x_test, y_test)
print(f"La precisión del modelo es: {accuracy:.2f}")

La precisión del modelo es: 0.63


Random Search:

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_dist_bagging = {
    'n_estimators': randint(100, 500),
    'max_samples': uniform(0.5, 0.5),      # Entre 0.5 y 1.0
    'max_features': uniform(0.5, 0.5),
    'bootstrap': [True, False],
    'estimator__max_depth': [None] + list(range(3, 30, 3)),
    'estimator__min_samples_split': randint(2, 20),
    'estimator__min_samples_leaf': randint(1, 10)}

random_bagging = RandomizedSearchCV(
    estimator=BaggingClassifier(estimator=DecisionTreeClassifier()),
    param_distributions=param_dist_bagging,
    n_iter=50,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=2,
    random_state=42)

random_bagging.fit(x_train, y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV] END bootstrap=True, estimator__max_depth=9, estimator__min_samples_leaf=8, estimator__min_samples_split=8, max_features=0.7229163764267956, max_samples=0.5499874579090014, n_estimators=187; total time=   1.0s
[CV] END bootstrap=True, estimator__max_depth=9, estimator__min_samples_leaf=8, estimator__min_samples_split=8, max_features=0.7229163764267956, max_samples=0.5499874579090014, n_estimators=187; total time=   1.1s
[CV] END bootstrap=True, estimator__max_depth=9, estimator__min_samples_leaf=8, estimator__min_samples_split=8, max_features=0.7229163764267956, max_samples=0.5499874579090014, n_estimators=187; total time=   1.0s
[CV] END bootstrap=True, estimator__max_depth=9, estimator__min_samples_leaf=8, estimator__min_samples_split=8, max_features=0.7229163764267956, max_samples=0.5499874579090014, n_estimators=187; total time=   1.1s
[CV] END bootstrap=True, estimator__max_depth=9, estimator__min_samples_leaf=8, es

,estimator,BaggingClassi...eClassifier())
,param_distributions,"{'bootstrap': [True, False], 'estimator__max_depth': [None, 3, ...], 'estimator__min_samples_leaf': <scipy.stats....t 0x176fe6b10>, 'estimator__min_samples_split': <scipy.stats....t 0x177121f90>, ...}"
,n_iter,50
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [ ]:
print("Mejores parámetros:", random_bagging.best_params_)

best_model_bag = random_bagging.best_estimator_

pred_bag = best_model_bag.predict(x_test)

accuracy_bag = best_model_bag.score(x_test, y_test)
print(f"La precisión del modelo es: {accuracy_bag:.2f}")

Mejores parámetros: {'bootstrap': False, 'estimator__max_depth': 18, 'estimator__min_samples_leaf': 2, 'estimator__min_samples_split': 4, 'max_features': np.float64(0.6504391549083848), 'max_samples': np.float64(0.6424202471887338), 'n_estimators': 180}
La precisión del modelo es: 0.62
